## What makes my grocery budget invisibly higher ?

This is my fifth year in Canada. In these 5 years, I have always used only 1 basket when shopping to control my total spending. While the basket is in perfect shape, the spending is not; I feel the grocery prices are increasing. However, when I look at the price of some of my 'must-buys' such as limes, I don't see much fluctuation. It seems those limes are determined to remain affordable!

In this project, I want to find out the story behind the grocery prices. Are they increasing? What is increasing? And why?

[here](https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1810024501&pickMembers%5B0%5D=1.6&cubeTimeFrame.startMonth=04&cubeTimeFrame.startYear=2019&cubeTimeFrame.endMonth=04&cubeTimeFrame.endYear=2024&referencePeriods=20190401%2C20240401) is where I got the dataset and the following is a small portion of it to have an idea of its schemas

In [1]:
import pandas as pd
import numpy as np
import mysql.connector

# connect to MySQL
connection = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    password="Xyz990603",
    database="monthly_average_retail"
)

cursor = connection.cursor()

#local file representation
base_path = r'/Users/stevenshi/Library/CloudStorage/GoogleDrive-shiyichen0404@gmail.com/My Drive/ML/Portfolio/data/Monthly_average_retail_prices_for_selected_products.csv'
df_base = pd.read_csv(base_path)

df_base.head(5)

ModuleNotFoundError: No module named 'mysql'

First, I want to understand the data at a macro level, so I calculated the total VALUE of purchasing one unit of each distinct Product on the same REF_DATE.

In [ ]:
query = """
WITH TOTAL_BUDGET AS (
	select
		REF_DATE,
		round(sum(VALUE),2) AS total
	from
		retail_price
	group by
	REF_DATE
	order by
	REF_DATE ASC
)

SELECT
	REF_DATE AS year_months,
    total AS total_budgets,
    round((total / LAG(total) OVER(ORDER BY REF_DATE) - 1), 2) AS budget_change
FROM 
	TOTAL_BUDGET
GROUP BY
	REF_DATE
order by
	REF_DATE
"""
cursor.execute(query)
results = cursor.fetchall()
df_monthly_total_budget_with_percentage_change = pd.DataFrame(results, columns=[column[0] for column in cursor.description])
df_monthly_total_budget_with_percentage_change.head(10)

,year_months,total_budgets,budget_change
0,2019-04,514.93,NaN
1,2019-05,536.12,0.04
2,2019-06,540.32,0.01
3,2019-07,529.07,-0.02
4,2019-08,522.82,-0.01
5,2019-09,510.28,-0.02
6,2019-10,499.13,-0.02
7,2019-11,521.88,0.05
8,2019-12,517.01,-0.01
9,2020-01,528.22,0.02


Now, I am able to plot the monthly total budget trend, but I also want to have an overview of the total change and the number of price-increase/decrease months over the past 5 years. Theoretically, DAX allows me to do this directly in Power BI, but I prefer using SQL.

In [ ]:
query = """
WITH first_last_date AS (
SELECT
(SELECT total
FROM monthly_total_budget_with_percentage_change
WHERE REF_DATE = (SELECT MAX(REF_DATE) FROM monthly_total_budget_with_percentage_change)
) AS latest_total,
(SELECT total
FROM monthly_total_budget_with_percentage_change
WHERE REF_DATE = (SELECT MIN(REF_DATE) FROM monthly_total_budget_with_percentage_change)
) AS earliest_total
)

select
	sum(case when percent_change > 0 then 1 else 0 end) as total_increase_num,
    sum(case when percent_change < 0 then 1 else 0 end) as total_decrease_num,
	round((latest_total - earliest_total) / earliest_total, 3) AS total_increase_percent
from
	monthly_total_budget_with_percentage_change, first_last_date
group by
	total_increase_percent
"""
cursor.execute(query)
results = cursor.fetchall()
df_increase_decrease_total_change = pd.DataFrame(results, columns=[column[0] for column in cursor.description])
df_increase_decrease_total_change

,total_increase_num,total_decrease_num,total_increase_percent
0,31,20,0.282


With this information, we can create a pie chart to visualize the increase and decrease, as well as the overall change.

After an overview at the macro level, I realize that grocery prices are indeed increasing. One of the most common questions is: which product has increased the most, and which has been the most stable? When it comes to fluctuation, one of the most fundamental and useful indices is the standard deviation. I would like to know the minimum, maximum, mean, and standard deviation for the price of each product over these five years. We will calculate them using SQL.

In [ ]:
query = """
WITH cor_date AS (
    SELECT 
        Products,
        VALUE,
        REF_DATE,
        ROW_NUMBER() OVER (PARTITION BY Products ORDER BY VALUE ASC) AS rn_min,
        ROW_NUMBER() OVER (PARTITION BY Products ORDER BY VALUE DESC) AS rn_max
    FROM 
        retail_price
)
SELECT 
    a.Products,
    ROUND(AVG(a.VALUE), 3) AS mean_product_price,
    ROUND(STDDEV(a.VALUE), 3) AS stdev_product_price,
    MIN(a.VALUE) AS min_product_price,
    MIN(CASE WHEN c.rn_min = 1 THEN c.REF_DATE END) AS min_price_date,
    MAX(a.VALUE) AS max_product_price,
    MAX(CASE WHEN c.rn_max = 1 THEN c.REF_DATE END) AS max_price_date
FROM 
    retail_price a
JOIN 
    cor_date c ON a.Products = c.Products
GROUP BY 
    a.Products
ORDER BY 
    stdev_product_price
"""
cursor.execute(query)
results = cursor.fetchall()
df_product_mean_stdev_min_max_date = pd.DataFrame(results, columns=[column[0] for column in cursor.description])
df_product_mean_stdev_min_max_date.head(20)

,Products,mean_product_price,stdev_product_price,min_product_price,min_price_date,max_product_price,max_price_date
0,"Bananas, per kilogram",1.445,0.035,1.39,2020-06,1.50,2023-06
1,"Lemons, unit",0.810,0.076,0.70,2019-07,1.00,2023-04
2,"Baby food, 128 millilitres",1.435,0.119,1.29,2019-04,1.70,2023-07
3,"Limes, unit",0.616,0.133,0.39,2019-07,0.89,2022-04
4,"Canned tuna, 170 grams",1.731,0.137,1.51,2019-11,2.07,2023-08
5,"Canned beans and lentils, 540 millilitres",1.376,0.169,1.14,2019-11,1.76,2023-12
6,"Canned corn, 341 millilitres",1.350,0.170,1.05,2019-10,1.70,2022-08
7,"Nut milk, 1.89 litres",3.872,0.180,3.54,2020-08,4.28,2023-04
8,"Canned pear, 398 millilitres",2.381,0.182,2.05,2019-05,2.74,2023-06
9,"Sunflower seeds, 400 grams",4.185,0.188,3.94,2022-01,4.67,2024-03


As you may see, I also included the corresponding dates for the minimum and maximum prices. We found that even when considering products with a fairly small standard deviation, most of the maximum prices appear in the latest years, which aligns with the total budget trend we discovered earlier.

In [ ]:
from IPython.display import IFrame

IFrame(src="https://drive.google.com/file/d/1HwqAwa8_n049Q6kQEUjukLMiDH3QzMhm/preview", width=800, height=400)

and [here](https://app.powerbi.com/groups/me/reports/a3e077d9-2fde-4a8a-a794-7316d9a58af0/8d050dd0fa73b0417722?experience=power-bi) is link to the Power BI page

## What happen to beef in these 5 years? 

One thing we can see from the analysis is that beef-related products all have comparatively high price fluctuations and significant price increases. What happened to them?

In [ ]:
path = r"C:\Users\Gaming Pc\Desktop\ML\Portfolio\data\Monthly_average_retail_prices_for_selected_products.csv"
df = pd.read_csv(path, delimiter=',')

In [ ]:
# select all the rows that contain beef as a Products string
df_beef = df[df['Products'].str.contains('beef',case=False)]
df_beef

,REF_DATE,GEO,DGUID,Products,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2019-04,Ontario,2016A000235,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1159447175,6.1,13.95,NaN,NaN,NaN,2
1,2019-05,Ontario,2016A000235,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1159447175,6.1,14.64,NaN,NaN,NaN,2
2,2019-06,Ontario,2016A000235,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1159447175,6.1,14.92,NaN,NaN,NaN,2
3,2019-07,Ontario,2016A000235,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1159447175,6.1,14.44,NaN,NaN,NaN,2
4,2019-08,Ontario,2016A000235,"Beef stewing cuts, per kilogram",Dollars,81,units,0,v1159447175,6.1,14.62,NaN,NaN,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300,2023-12,Ontario,2016A000235,"Ground beef, per kilogram",Dollars,81,units,0,v1159447178,6.4,12.23,NaN,NaN,NaN,2
301,2024-01,Ontario,2016A000235,"Ground beef, per kilogram",Dollars,81,units,0,v1159447178,6.4,11.06,NaN,NaN,NaN,2
302,2024-02,Ontario,2016A000235,"Ground beef, per kilogram",Dollars,81,units,0,v1159447178,6.4,11.09,NaN,NaN,NaN,2
303,2024-03,Ontario,2016A000235,"Ground beef, per kilogram",Dollars,81,units,0,v1159447178,6.4,11.49,NaN,NaN,NaN,2


we have many useless columns here, let's do some cleaning

In [ ]:
list(df_beef.columns)

['REF_DATE',
 'GEO',
 'DGUID',
 'Products',
 'UOM',
 'UOM_ID',
 'SCALAR_FACTOR',
 'SCALAR_ID',
 'VECTOR',
 'COORDINATE',
 'VALUE',
 'STATUS',
 'SYMBOL',
 'TERMINATED',
 'DECIMALS']

In [ ]:
col_todrop = ['GEO','DGUID','UOM','UOM_ID','SCALAR_FACTOR','SCALAR_ID','VECTOR', 'COORDINATE','STATUS','SYMBOL', 'TERMINATED','DECIMALS']
df_beef = df_beef.drop(columns=col_todrop)

In [ ]:
#rename the column for consistancy
df_beef.rename(columns={'REF_DATE':'Date','VALUE':'Value'}, inplace=True)

since all beef products share the same unit, we can just remove it

In [ ]:
df_beef['Products'] = df_beef['Products'].str.replace(', per kilogram','')

In [ ]:
df_beef

,Date,Products,Value
0,2019-04,Beef stewing cuts,13.95
1,2019-05,Beef stewing cuts,14.64
2,2019-06,Beef stewing cuts,14.92
3,2019-07,Beef stewing cuts,14.44
4,2019-08,Beef stewing cuts,14.62
...,...,...,...
300,2023-12,Ground beef,12.23
301,2024-01,Ground beef,11.06
302,2024-02,Ground beef,11.09
303,2024-03,Ground beef,11.49


In [ ]:
list(df_beef['Products'].unique())

['Beef stewing cuts',
 'Beef striploin cuts',
 'Beef top sirloin cuts',
 'Beef rib cuts',
 'Ground beef']

In [ ]:
pivoted_data = df_beef.pivot_table(index='Date', columns='Products', values= 'Value')
pivoted_data['Total value'] = pivoted_data.sum(axis=1)
pivoted_data['Total value change'] = pivoted_data['Total value'].pct_change()
pivoted_data['Beef rib cuts change'] = pivoted_data['Beef rib cuts'].pct_change()
pivoted_data['Beef stewing cuts change'] = pivoted_data['Beef stewing cuts'].pct_change()
pivoted_data['Beef striploin cuts change'] = pivoted_data['Beef striploin cuts'].pct_change()
pivoted_data['Beef top sirloin cuts change'] = pivoted_data['Beef top sirloin cuts'].pct_change()
pivoted_data['Ground beef cuts change'] = pivoted_data['Ground beef'].pct_change()


In [ ]:
pivoted_data.head()

Products,Beef rib cuts,Beef stewing cuts,Beef striploin cuts,Beef top sirloin cuts,Ground beef,Total value,Total value change,Beef rib cuts change,Beef stewing cuts change,Beef striploin cuts change,Beef top sirloin cuts change,Ground beef cuts change
Date,,,,,,,,,,,,
2019-04,16.48,13.95,15.94,13.58,9.32,69.27,NaN,NaN,NaN,NaN,NaN,NaN
2019-05,23.76,14.64,17.99,16.82,9.33,82.54,0.191569,0.441748,0.049462,0.128607,0.238586,0.001073
2019-06,21.14,14.92,19.19,15.68,9.53,80.46,-0.025200,-0.110269,0.019126,0.066704,-0.067776,0.021436
2019-07,23.05,14.44,19.26,14.97,9.45,81.17,0.008824,0.090350,-0.032172,0.003648,-0.045281,-0.008395
2019-08,22.98,14.62,18.42,12.36,9.73,78.11,-0.037699,-0.003037,0.012465,-0.043614,-0.174349,0.029630


In [ ]:
pivoted_data.to_csv('pirovted_beef_data.csv',index=True)

In [ ]:
IFrame('https://drive.google.com/file/d/1CD0RnQzDQmMfzt4uFoab0G-Iaalsz_6f/preview', width=800, height=400)

and [here](https://app.powerbi.com/groups/me/reports/a3e077d9-2fde-4a8a-a794-7316d9a58af0/8d050dd0fa73b0417722?experience=power-bi) is link to the Power BI page, please go to page 'Beef Products'

There are two interesting factors I would like to mention:
1. According to 'Beef Products Price Fluctuation', compared to Beef Rib, Striploin Cuts, and Top Sirloin Cuts, **ground beef** and **beef stewing** have much more stable prices.
2. Both 'Beef Products Total Price and Price Change' and 'Monthly Average Beef Price Change' (table at the lower left) suggest a significant amount of price change between **December** and the upcoming **January**

I believe a reasonable next step is to go to local cattle feeders and ask them what happens between December and January. Here are some questions that can be asked:
1. Is there a beef supply shortage at the very beginning of each year? If yes, why?
2. Is there an increase in the cost of cattle farming at the beginning of each year? What is the main issue?
3. What happens at the very end of each year? Is there a supply surplus?


## Conclusion

In this project, we analyzed 5 years of grocery price data in Ontario, Canada to understand trends and drivers of increasing food costs. The analysis revealed several key insights:

1. Overall grocery budgets have increased by 28.2% from 2019 to 2024, with 31 months seeing price increases compared to 20 months with decreases. This indicates a clear inflationary trend in food prices over the 5 year period analyzed.
2. Across all products analyzed, the most stable prices were for bananas, lemons, baby food, limes, canned tuna, and canned beans/lentils. In contrast, beef rib, infant formula, striploin, sirloin and vegetable oil saw the highest price variability.
3. Focusing on beef products, the analysis found that ground beef and beef stewing cuts had much more stable prices compared to premium cuts like rib, striploin and top sirloin. So if you are saving and want to eat beef, ground beef and beef stew can be a great choice.
4. A pattern emerged where beef prices tend to decrease dramatically (especialy premium cuts like rib, striploin and top sirloin) in December compared to the yearly average, then rebound in January. Potential factors include a beef supply shortage at the start of each year, increased cattle farming costs, and a supply surplus at year-end.

And here is a abstract mindmap used for this project

In [ ]:
IFrame('https://drive.google.com/file/d/1oEo0C-F2EfTC3UT7TS192qJVVVni_GNL/preview', width=800, height=400)